# DQMBot — Batch Image Query Driver

**Image layout expected:**
```
images/
    <plotName>/
        <plotName>_run<XXXXXX>.png
```

**Output layout (with run_id — preserves previous runs):**
```
results/
    <run_id>/
        <plotName>/
            <plotName>_<model>_run<XXXXXX>.txt
        summary_<run_id>.csv
```

**Output layout (no run_id — overwrites):**
```
results/
    <plotName>/
        <plotName>_<model>_run<XXXXXX>.txt
    summary.csv
```

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from owui_client import (
    list_models, get_knowledge_map, query,
    batch_query_images, _collect_images, resolve_output_dir, resolve_output_file,
)

print('owui_client loaded OK')

owui_client loaded OK


In [2]:
# ── Discover available models and knowledge collections ───────────────────────
print('=== Models ===')
for m in list_models():
    print(' ', m)

print()
print('=== Knowledge collections ===')
kb_map = get_knowledge_map()
for name, kid in kb_map.items():
    print(f'  {name:30s}  {kid}')

=== Models ===
  835
  acnet-documentation
  acnet-documentation-coder
  dqm-chatbot
  gemma3:latest
  qwen2.5vl:32b
  qwen2.5vl:latest
  qwen3-vl:latest
  erlang-otp
  litellm-ow.qwen/qwen3-coder-next
  qwen3-vl:32b
  srf-data-test
  stockroom
  aeolus
  gpt-oss:20b
  qwen2.5:7b
  stockroom-clone-hayden
  stockroom-clone-hayden-medium
  litellm-ow.qwen/qwen35-9b
  qwenqwen35-9b-no-thinking
  nonfree.azure/gpt-5-nano
  nonfree.azure/gpt-5.2
  nonfree.azure/auto
  vllm.gpt-oss:120b
  litellm-ow.google/gemma4-31b
  litellm-ow.qwen/qwen3.6
  policies

=== Knowledge collections ===
  ECFR                            01120bf5-3580-49d4-8bc5-4964b7d96cef
  FESHM                           281b78b3-3cb2-4cfe-ab4b-9e7547713f82
  Fermilab Policies               ff41ce28-5f46-4f3a-a423-310fd63a21fd
  Rookie Books                    c0de275c-458a-44f7-b800-834c6e52eb1a
  Erlang OTP-26.2.5.11            b18bde0e-da9c-45e6-8cbb-5f4fe7400b5d
  AEOLUS Codebase                 cb71bb35-e24b-46f9-b31f-f9

In [4]:
# ── Configuration ─────────────────────────────────────────────────────────────

IMAGE_ROOT  = Path('images')
OUTPUT_ROOT = Path('results')

# Set a string to preserve previous runs alongside this one.
# Leave as None to overwrite.
RUN_ID = 'baseline'
# RUN_ID = None

MODELS = [
 'qwen2.5vl:latest',            #7b    
 'qwen2.5vl:32b',               #32b
 'qwen3-vl:latest',             #8b
 'litellm-ow.qwen/qwen3.6',     #35b 
 'gemma3:latest',               #4b
 'litellm-ow.google/gemma4-31b',#31b
]


COLLECTIONS = [
    kb_map['DQM shift rules'],
]

SYSTEM_PROMPT = (
 """\
You are an assistant to shifters of the CMS experiment during detector operations.
You are tasked to judge if a input plot is good or bad.
 
In your output, make 4 sections:
 - Quote the relevant section of instructions for the input plot
 - Describe the input plot
 - Compare input plot to the instruction
 - Decide if the plot is good or bad\
"""
)

PROMPT = (
    ''
)

DELAY = 1.5
# ──────────────────────────────────────────────────────────────────────────────

In [5]:
# ── Sanity check: show what will be processed and where it will land ──────────
pairs = _collect_images(IMAGE_ROOT, ('.png', '.jpg', '.jpeg', '.webp'))
plot_names = sorted(set(p for p, _ in pairs))

print(f'Plots found  : {len(plot_names)}')
for pn in plot_names:
    imgs = [img for p, img in pairs if p == pn]
    print(f'  {pn}/  ({len(imgs)} images)')
    for img in imgs:
        print(f'    {img.name}')

print(f'\nModels       : {len(MODELS)}')
for m in MODELS:
    print(f'  {m}')

print(f'\nRun ID       : {RUN_ID or "(none — overwrite mode)"}')
print(f'Total queries: {len(pairs) * len(MODELS)}')

print('\nExample output paths:')
for model in MODELS:
    plot_name, img = pairs[0]
    d = resolve_output_dir(OUTPUT_ROOT, plot_name, RUN_ID)
    f = resolve_output_file(d, img, model)
    print(f'  {f}')

Plots found  : 1
  HCALoccupancy_run404150/  (1 images)
    HCALoccupancy_run404150.png

Models       : 6
  qwen2.5vl:latest
  qwen2.5vl:32b
  qwen3-vl:latest
  litellm-ow.qwen/qwen3.6
  gemma3:latest
  litellm-ow.google/gemma4-31b

Run ID       : baseline
Total queries: 6

Example output paths:
  results/baseline/HCALoccupancy_run404150/HCALoccupancy_run404150_qwen2.5vl_latest.txt
  results/baseline/HCALoccupancy_run404150/HCALoccupancy_run404150_qwen2.5vl_32b.txt
  results/baseline/HCALoccupancy_run404150/HCALoccupancy_run404150_qwen3-vl_latest.txt
  results/baseline/HCALoccupancy_run404150/HCALoccupancy_run404150_litellm-ow.qwen_qwen3.6.txt
  results/baseline/HCALoccupancy_run404150/HCALoccupancy_run404150_gemma3_latest.txt
  results/baseline/HCALoccupancy_run404150/HCALoccupancy_run404150_litellm-ow.google_gemma4-31b.txt


In [6]:
# ── Smoke test: one image, first model ───────────────────────────────────────
if pairs:
    plot_name, img = pairs[0]
    test = query(
        PROMPT,
        model=MODELS[0],
        system=SYSTEM_PROMPT,
        image_path=img,
        collection_ids=COLLECTIONS,
    )
    print(f"Plot    : {plot_name}")
    print(f"Model   : {test['model_used']}")
    print(f"Image   : {test['image']}")
    print(f"Latency : {test['latency_s']}s")
    print(f"Error   : {test['error']}")
    print()
    print(test['response'])

Plot    : HCALoccupancy_run404150
Model   : qwen2.5vl:latest
Image   : images/HCALoccupancy_run404150.png
Latency : 15.56s
Error   : None

### Instructions for the Input Plot:
The plot should show the HCal TP ET-weighted Occupancy at Layer1, with entries counted and a color scale indicating the number of entries per bin.

### Description of the Input Plot:
The plot is a 2D histogram representing the HCal TP ET-weighted Occupancy at Layer1. The x-axis represents the iPhi (phi coordinate), and the y-axis represents the iEta (eta coordinate). The color scale on the right indicates the number of entries per bin, ranging from 0 to 6000. The histogram shows a distribution of occupancy values across the phi and eta coordinates, with varying intensities of color indicating the number of entries.

### Comparison to the Instruction:
The plot aligns well with the instructions. It clearly shows the HCal TP ET-weighted Occupancy at Layer1, with the x-axis representing iPhi and the y-axis representi

In [7]:
# ── Full batch ────────────────────────────────────────────────────────────────
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

results = batch_query_images(
    PROMPT,
    image_root=IMAGE_ROOT,
    models=MODELS,
    output_root=OUTPUT_ROOT,
    run_id=RUN_ID,
    system=SYSTEM_PROMPT,
    collection_ids=COLLECTIONS,
    delay=DELAY,
    verbose=True,
)

print(f'\nDone. {len(results)} queries completed.')

[1/6] model=qwen2.5vl:latest  plot=HCALoccupancy_run404150  image=HCALoccupancy_run404150.png ... ERROR
[2/6] model=qwen2.5vl:32b  plot=HCALoccupancy_run404150  image=HCALoccupancy_run404150.png ... ERROR
[3/6] model=qwen3-vl:latest  plot=HCALoccupancy_run404150  image=HCALoccupancy_run404150.png ... 72.05s → results/baseline/HCALoccupancy_run404150/HCALoccupancy_run404150_qwen3-vl_latest.txt
[4/6] model=litellm-ow.qwen/qwen3.6  plot=HCALoccupancy_run404150  image=HCALoccupancy_run404150.png ... 8.99s → results/baseline/HCALoccupancy_run404150/HCALoccupancy_run404150_litellm-ow.qwen_qwen3.6.txt
[5/6] model=gemma3:latest  plot=HCALoccupancy_run404150  image=HCALoccupancy_run404150.png ... 9.79s → results/baseline/HCALoccupancy_run404150/HCALoccupancy_run404150_gemma3_latest.txt
[6/6] model=litellm-ow.google/gemma4-31b  plot=HCALoccupancy_run404150  image=HCALoccupancy_run404150.png ... 8.56s → results/baseline/HCALoccupancy_run404150/HCALoccupancy_run404150_litellm-ow.google_gemma4-31b.

In [8]:
# ── Summary ───────────────────────────────────────────────────────────────────
df = pd.DataFrame(results)
df['image_name'] = df['image'].apply(lambda p: Path(p).name if p else None)

errors = df[df['error'].notna()]
if not errors.empty:
    print(f'WARNING: {len(errors)} failed queries:')
    display(errors[['plot_name', 'model_used', 'image_name', 'error']])
else:
    print('All queries succeeded.')

print()
display(
    df.groupby(['plot_name', 'model_used'])['latency_s']
      .agg(['count', 'mean', 'min', 'max'])
      .round(2)
      .rename(columns={'count': 'n', 'mean': 'avg_s', 'min': 'min_s', 'max': 'max_s'})
)

,plot_name,model_used,image_name,error
0,HCALoccupancy_run404150,qwen2.5vl:latest,HCALoccupancy_run404150.png,"HTTPSConnectionPool(host='openwebui.fnal.gov',..."
1,HCALoccupancy_run404150,qwen2.5vl:32b,HCALoccupancy_run404150.png,"HTTPSConnectionPool(host='openwebui.fnal.gov',..."


n   avg_s   min_s   max_s
plot_name               model_used                                  
HCALoccupancy_run404150 gemma3:latest      1    9.79    9.79    9.79
                        google/gemma4-31b  1    8.56    8.56    8.56
                        qwen/qwen3.6       1    8.99    8.99    8.99
                        qwen2.5vl:32b      1  120.13  120.13  120.13
                        qwen2.5vl:latest   1  120.08  120.08  120.08
                        qwen3-vl:latest    1   72.05   72.05   72.05

In [9]:
# ── Save CSV next to the run's output folder ──────────────────────────────────
if RUN_ID:
    csv_path = OUTPUT_ROOT / RUN_ID / f'summary_{RUN_ID}.csv'
else:
    csv_path = OUTPUT_ROOT / 'summary.csv'

df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')

# Show result tree
print()
root = OUTPUT_ROOT / RUN_ID if RUN_ID else OUTPUT_ROOT
for item in sorted(root.iterdir()):
    if item.is_dir():
        txts = list(item.glob('*.txt'))
        print(f'  {item.name}/  ({len(txts)} files)')
        for t in sorted(txts):
            print(f'    {t.name}')
    elif item.suffix == '.csv':
        print(f'  {item.name}')

Saved: results/baseline/summary_baseline.csv

  HCALoccupancy_run404150/  (6 files)
    HCALoccupancy_run404150_gemma3_latest.txt
    HCALoccupancy_run404150_litellm-ow.google_gemma4-31b.txt
    HCALoccupancy_run404150_litellm-ow.qwen_qwen3.6.txt
    HCALoccupancy_run404150_qwen2.5vl_32b.txt
    HCALoccupancy_run404150_qwen2.5vl_latest.txt
    HCALoccupancy_run404150_qwen3-vl_latest.txt
  summary_baseline.csv


In [10]:
# ── Side-by-side comparison: one image across all models ─────────────────────
COMPARE_PLOT  = plot_names[0]
COMPARE_IMAGE = pairs[0][1].name

subset = df[(df['plot_name'] == COMPARE_PLOT) & (df['image_name'] == COMPARE_IMAGE)]
for _, row in subset.iterrows():
    print('=' * 72)
    print(f"Model   : {row['model_used']}")
    print(f"Latency : {row['latency_s']}s")
    print()
    print(row['error'] and f"ERROR: {row['error']}" or row['response'])
    print()

Model   : qwen2.5vl:latest
Latency : 120.08s

ERROR: HTTPSConnectionPool(host='openwebui.fnal.gov', port=443): Read timed out. (read timeout=120)

Model   : qwen2.5vl:32b
Latency : 120.13s

ERROR: HTTPSConnectionPool(host='openwebui.fnal.gov', port=443): Read timed out. (read timeout=120)

Model   : qwen3-vl:latest
Latency : 72.05s

### Quote the relevant section of instructions for the input plot  
*"For HCAL TP occupancy plots, ensure no large areas of unexpected dead zones (zero occupancy) exist, as these may indicate hardware issues or misconfigurations in the detector system. Expected occupancy patterns should show uniformity with minor deviations due to detector design, not abrupt gaps."*  

### Describe the input plot  
The plot is a 2D heat map titled *"HCal TP ET-weighted Occupancy at Layer1"*, with **iPhi** (y-axis, 10–70) vs. **iEta** (x-axis, -40–40). It uses a color scale where purple = 0 entries, blue = ~1000 entries, green = ~3000 entries, yellow = ~4000 entries, and red